# Distribution Grid with Neural Network Forecasters

This example demonstrates how **neural-network-based prediction models** can
serve as simulators in an energysim co-simulation.  Each load and generator
in a distribution network is modelled by a small feedforward neural network
that predicts power based on time-of-day, weather, and calendar features.

| Simulator | Type | Step Size | Model |
|---|---|---|---|
| Feature data | CSV | 900 s | Weather + time features |
| Residential load | NN (ext) | 60 s | 4→16→8→1 feedforward |
| Commercial load | NN (ext) | 60 s | 4→16→8→1 feedforward |
| Solar park | NN (ext) | 60 s | 4→16→8→1 feedforward |
| Wind farm | NN (ext) | 60 s | 4→16→8→1 feedforward |
| Distribution grid | Pandapower | 300 s | AC power flow |

In [ ]:
import os, sys
from pathlib import Path

HERE = Path(os.getcwd()).resolve()
_src = str(HERE.parents[1] / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
os.chdir(HERE)

import numpy as np
import matplotlib.pyplot as plt
from energysim import world

print('energysim loaded from:', _src)

## Generate Data & Train Neural Networks

First we create synthetic weather/time features and train four small
feedforward neural networks on characteristic load/generation patterns.

In [ ]:
import generate_data
generate_data.main()

import train_models
train_models.main()

## Set Up Co-Simulation

In [ ]:
SIM_DIR = str(HERE / 'simulators')

my_world = world(start_time=0, stop_time=86400, logging=True, t_macro=900)

# Feature data (weather + time)
my_world.add_simulator(sim_type='csv', sim_name='features',
    sim_loc=str(HERE / 'features.csv'),
    outputs=['hour_sin', 'hour_cos', 'temperature',
             'cloud_cover', 'wind_speed', 'is_weekend'],
    step_size=900)

# NN-based load/generation forecasters
my_world.add_simulator(sim_type='external', sim_name='nn_residential',
    sim_loc=SIM_DIR, inputs=['hour_sin','hour_cos','temperature','is_weekend'],
    outputs=['P_load'], step_size=60)

my_world.add_simulator(sim_type='external', sim_name='nn_commercial',
    sim_loc=SIM_DIR, inputs=['hour_sin','hour_cos','temperature','is_weekend'],
    outputs=['P_load'], step_size=60)

my_world.add_simulator(sim_type='external', sim_name='nn_pv',
    sim_loc=SIM_DIR, inputs=['hour_sin','hour_cos','cloud_cover','temperature'],
    outputs=['P_gen'], step_size=60)

my_world.add_simulator(sim_type='external', sim_name='nn_wind',
    sim_loc=SIM_DIR, inputs=['hour_sin','hour_cos','wind_speed','temperature'],
    outputs=['P_gen'], step_size=60)

# Pandapower distribution grid
my_world.add_simulator(sim_type='powerflow', sim_name='grid',
    sim_loc=str(HERE / 'distribution_grid.p'),
    inputs=['Residential.p_mw', 'Commercial.p_mw',
            'SolarPark.p_mw', 'WindFarm.p_mw'],
    outputs=['HV_Bus.vm_pu', 'MV_Bus1.vm_pu', 'MV_Bus2.vm_pu',
             'MV_Bus3.vm_pu', 'MV_Bus4.vm_pu',
             'Residential.p_mw', 'Commercial.p_mw',
             'SolarPark.p_mw', 'WindFarm.p_mw'],
    step_size=300, pf='pf')

print('All simulators added.')

## Connections

Fan-out connections distribute feature variables to all four NN models
simultaneously.  NN outputs feed directly into the pandapower grid.

In [ ]:
connections = {
    # Features -> NN models
    'features.hour_sin':     ('nn_residential.hour_sin', 'nn_commercial.hour_sin',
                               'nn_pv.hour_sin', 'nn_wind.hour_sin'),
    'features.hour_cos':     ('nn_residential.hour_cos', 'nn_commercial.hour_cos',
                               'nn_pv.hour_cos', 'nn_wind.hour_cos'),
    'features.temperature':  ('nn_residential.temperature', 'nn_commercial.temperature',
                               'nn_pv.temperature', 'nn_wind.temperature'),
    'features.is_weekend':   ('nn_residential.is_weekend', 'nn_commercial.is_weekend'),
    'features.cloud_cover':  'nn_pv.cloud_cover',
    'features.wind_speed':   'nn_wind.wind_speed',
    # NN outputs -> grid
    'nn_residential.P_load': 'grid.Residential.p_mw',
    'nn_commercial.P_load':  'grid.Commercial.p_mw',
    'nn_pv.P_gen':           'grid.SolarPark.p_mw',
    'nn_wind.P_gen':         'grid.WindFarm.p_mw',
}
my_world.add_connections(connections)

## Run Simulation

In [ ]:
my_world.simulate(pbar=True, record_all=False)

results = my_world.results(to_csv=False)
for name, df in results.items():
    print(f'{name:15s} -> {df.shape[0]} steps, {df.shape[1]-1} variables')

## Analyse Results

In [ ]:
grid_df = results['grid']
hours = grid_df['time'] / 3600.0

fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

# 1. Bus voltages
ax = axes[0]
for bus in ['HV_Bus.vm_pu', 'MV_Bus1.vm_pu', 'MV_Bus2.vm_pu',
            'MV_Bus3.vm_pu', 'MV_Bus4.vm_pu']:
    if bus in grid_df.columns:
        ax.plot(hours, grid_df[bus], label=bus.split('.')[0])
ax.set_ylabel('Voltage (pu)')
ax.set_title('Bus Voltages')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Load profiles
ax = axes[1]
if 'Residential.p_mw' in grid_df.columns:
    ax.plot(hours, grid_df['Residential.p_mw'] * 1000, label='Residential')
if 'Commercial.p_mw' in grid_df.columns:
    ax.plot(hours, grid_df['Commercial.p_mw'] * 1000, label='Commercial')
ax.set_ylabel('Load (kW)')
ax.set_title('Load Profiles (from NN predictors)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Generation profiles
ax = axes[2]
if 'SolarPark.p_mw' in grid_df.columns:
    ax.plot(hours, grid_df['SolarPark.p_mw'] * 1000, label='Solar PV')
if 'WindFarm.p_mw' in grid_df.columns:
    ax.plot(hours, grid_df['WindFarm.p_mw'] * 1000, label='Wind')
ax.set_ylabel('Generation (kW)')
ax.set_title('Generation Profiles (from NN predictors)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4. Net import (loads - generation)
ax = axes[3]
load_total = np.zeros(len(grid_df))
gen_total = np.zeros(len(grid_df))
for col in ['Residential.p_mw', 'Commercial.p_mw']:
    if col in grid_df.columns:
        load_total += grid_df[col].values
for col in ['SolarPark.p_mw', 'WindFarm.p_mw']:
    if col in grid_df.columns:
        gen_total += grid_df[col].values
net_import = (load_total - gen_total) * 1000
ax.fill_between(hours, net_import, alpha=0.4, label='Net import')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel('Net Import (kW)')
ax.set_xlabel('Time (h)')
ax.set_title('Grid Import / Export')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ml_grid_results.png', dpi=150)
plt.show()
print('Figure saved to ml_grid_results.png')

## Summary

**Key observations:**

- The **residential load** peaks in the morning (~8 am) and evening (~7 pm),
  reflecting typical household patterns.
- The **commercial load** follows working hours (9–17), dropping sharply on
  weekends (this run uses a weekday scenario).
- **PV generation** follows a bell curve centred around noon, modulated by
  cloud cover.
- **Wind generation** varies with wind speed following a cubic power curve.
- **Bus voltages** remain close to 1.0 pu throughout, as the loads and
  generation are small relative to the 40 MVA transformer rating.
- **Net grid import** is positive when loads exceed local generation and
  negative during periods of high renewable output — demonstrating
  bidirectional power flow in the distribution network.

This example shows how lightweight ML models (numpy-only neural networks)
can serve as drop-in simulators within energysim’s co-simulation framework,
enabling data-driven forecasting alongside physics-based power-flow analysis.